# Practical No. 05 — Vector Semantics and Word Embeddings

**Name:** Krishna Mungase  
**PRN:** 202301040020

---

Vector Semantics represents words as dense numerical vectors so that semantic relationships between words become geometric relationships between vectors.

> *"You shall know a word by the company it keeps."* — J. R. Firth

**Word2Vec** is a shallow neural model that learns these vectors from context.
- **CBOW**: predict the target word from its context.
- **Skip-gram**: predict the context from the target word.

**Steps:** Prepare corpus → Train Word2Vec → Inspect vectors → Find similar words → Visualize with t-SNE.

## 1. Imports and Corpus

A tiny toy corpus grouped around three themes (royalty, animals, tech) so that Word2Vec has enough co-occurrence signal to produce meaningful similarities.

In [ ]:
from gensim.models import Word2Vec
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
sentences = [
    ["king", "queen", "royal", "palace"],
    ["king", "queen", "rule", "kingdom"],
    ["king", "queen", "crown", "throne"],
    ["prince", "princess", "royal", "palace"],
    ["prince", "princess", "castle", "throne"],
    ["king", "prince", "royal", "family"],
    ["queen", "princess", "royal", "family"],
    ["man", "woman", "husband", "wife"],
    ["boy", "girl", "young", "child"],

    ["cat", "dog", "pets", "animals"],
    ["cat", "dog", "home", "animals"],
    ["cat", "kitten", "milk", "pets"],
    ["dog", "puppy", "bone", "pets"],
    ["cat", "sits", "mat", "pets"],
    ["dog", "sits", "mat", "pets"],
    ["cat", "dog", "home", "garden"],

    ["python", "java", "programming", "language"],
    ["python", "java", "code", "software"],
    ["python", "programming", "language", "code"],
    ["java", "programming", "language", "code"],
    ["computer", "laptop", "program", "software"],
    ["computer", "laptop", "machine", "hardware"],
    ["machine", "learning", "python", "code"],
    ["deep", "learning", "python", "neural"],
]

print(f"Number of sentences: {len(sentences)}")

## 2. Train Word2Vec

**Parameters:**
- `vector_size` — dimension of each word vector
- `window` — number of context words on each side
- `min_count` — ignore words that appear fewer times than this
- `sg=0` → CBOW, `sg=1` → Skip-gram

In [ ]:
model = Word2Vec(
    sentences=sentences,
    vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    epochs=500,
    seed=42,
)

print("Vocabulary size:", len(model.wv))
print("Vector for 'king' (first 10 dims):", model.wv["king"][:10])

## 3. Most Similar Words

Cosine similarity between word vectors finds the closest neighbours of a given word.

In [ ]:
for word in ["king", "cat", "python"]:
    print(f"Most similar to '{word}':")
    for neighbour, score in model.wv.most_similar(word, topn=5):
        print(f"  {neighbour:<12} {score:.4f}")
    print()

## 4. Pairwise Similarity

Direct cosine similarity between two words.

In [ ]:
pairs = [("king", "queen"), ("cat", "dog"), ("python", "java"), ("king", "cat")]

for w1, w2 in pairs:
    sim = model.wv.similarity(w1, w2)
    print(f"similarity({w1}, {w2}) = {sim:.4f}")

## 5. Visualize Embeddings with t-SNE

Word vectors live in a 50-dimensional space. t-SNE projects them down to 2D so we can plot them — semantically similar words should cluster together.

In [ ]:
words = list(model.wv.index_to_key)
vectors = np.array([model.wv[w] for w in words])

tsne = TSNE(n_components=2, perplexity=5, random_state=42, init="pca")
coords = tsne.fit_transform(vectors)

plt.figure(figsize=(10, 7))
plt.scatter(coords[:, 0], coords[:, 1], color="steelblue")

for i, word in enumerate(words):
    plt.annotate(word, (coords[i, 0], coords[i, 1]), fontsize=9,
                 xytext=(4, 4), textcoords="offset points")

plt.title("Word2Vec Embeddings visualized with t-SNE")
plt.xlabel("t-SNE dim 1")
plt.ylabel("t-SNE dim 2")
plt.tight_layout()
plt.show()

## Conclusion

- Word2Vec turns words into dense vectors that encode distributional meaning.
- Words sharing contexts (`king`/`queen`, `cat`/`dog`, `python`/`java`) end up with high cosine similarity and cluster together in the t-SNE plot.
- **Advantages:** captures semantic similarity, efficient, supports vector arithmetic.
- **Limitations:** a single vector per word (no sense disambiguation), needs large corpora for quality, and cannot represent words unseen during training.